In [13]:
from functools import cache
from pathlib import Path

import pandas as pd
import google.auth
import geopandas as gpd

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.sql import get_engine

# from shared_utils import bq_utils
# Initialize credentials and DB engine
credentials, project = google.auth.default()
db_engine = get_engine()

import _prep_crosswalk_ntd
import _load_gtfs_data
from update_vars import DIGEST_DICT

In [14]:
from update_vars import DIGEST_DICT as GTFS_DATA_DICT, abbrev_month, analysis_month 

In [15]:
@cache
def gcs_pandas():
    return GCSPandas()

In [16]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

## Operator = Why is October 2026 stuff missing? 

In [17]:
GCS_PATH = f"{GTFS_DATA_DICT.dir}processed/"

In [18]:
analysis_name = "Alameda-Contra Costa Transit District"

In [19]:
# Dataset 1
fct_monthly_routes_url = f"{GCS_PATH}{GTFS_DATA_DICT.route_map}.parquet"
fct_monthly_route_df = gpd.read_parquet(fct_monthly_routes_url,
                                        filters=[[("Analysis Name", "==", analysis_name)]],
                                        storage_options={"token": credentials.token}).reset_index()
fct_monthly_route_df['Number'] = fct_monthly_route_df.index

In [21]:
fct_monthly_route_df["Month First Day"].unique()

array([datetime.date(2026, 10, 1)], dtype=object)

In [22]:
# Dataset 2
schedule_rt_route_direction_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.schedule_rt_route_direction}.parquet"
schedule_rt_route_direction_summary_df = gcs_pandas().read_parquet(schedule_rt_route_direction_summary_url,
                         filters=[[("Analysis Name", "==", analysis_name)]])
one_month_schedule_rt_route_direction_summary_df = (schedule_rt_route_direction_summary_df.loc[schedule_rt_route_direction_summary_df["Day Type"] == "Weekday"]
    .sort_values(by = ["Date", "Route"], ascending = [False, True])
    .drop_duplicates(subset = ["Route"])
                                                   )

In [23]:
one_month_schedule_rt_route_direction_summary_df["Date"].unique()

<DatetimeArray>
['2026-11-01 00:00:00']
Length: 1, dtype: datetime64[ns]

In [24]:
# Dataset 3 
operator_hourly_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.hourly_day_type_summary}.parquet"
operator_hourly_summary_df = gcs_pandas().read_parquet(operator_hourly_summary_url,
                                             filters=[[("Analysis Name", "==", analysis_name),
                                                      ("Departure Hour", "<=", 24)]])

In [26]:
operator_hourly_summary_df["Date"].unique()

array(['10-2025', '11-2025', '12-2025', '01-2026', '02-2026', '03-2026',
       '04-2026', '05-2026', '06-2026', '07-2026', '08-2026', '09-2026',
       '10-2026'], dtype=object)

In [27]:
# Dataset 5 
operator_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.operator_summary}.parquet"
operator_df = (gcs_pandas().read_parquet(
    operator_summary_url,
    filters=[
        ("Day Type", "==", "Weekday"),
        ("Analysis Name", "==", analysis_name),
        ("VP Name", "!=", "None")],
)).sort_values(by = ["Date","Daily TU Trips"], ascending = [False,False]).drop_duplicates(subset = [
    "Analysis Name","Date"
])
operator_df["Pct TU Trips"] = operator_df["Pct TU Trips"].round(2)

In [28]:
operator_df["Date"].unique()

<DatetimeArray>
['2026-11-01 00:00:00', '2026-10-01 00:00:00', '2026-09-01 00:00:00']
Length: 3, dtype: datetime64[ns]

## Caltrans District, GTFS by Operator is wrong for D4

In [39]:
from update_vars import DIGEST_DICT, PROCESSED_GCS
import gcsfs
import _ct_report_utils as utils

In [30]:
district = "04 - Bay Area / Oakland"

In [33]:
f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_operator_summary.parquet'

In [34]:
f"{GCS_PATH}{GTFS_DATA_DICT.operator_summary}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_operator_summary.parquet'

In [37]:
# Dataset 4
operator_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

operator_df = pd.read_parquet(
    operator_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[
        ("Caltrans District", "==", district),
        ("Day Type", "==", "Weekday")
    ],
).sort_values(by = ["Analysis Name", "Date"], ascending = [False,False]).drop_duplicates(subset = ["Analysis Name"])

In [38]:
operator_df

,Date,Analysis Name,Caltrans District,VP Name,TU Name,N Trips,Day Type,Daily Trips,Ttl Service Hours,N Routes,N Days,N Shapes,N Stops,Daily Arrivals,VP Messages Per Minute,N VP Trips,Daily VP Trips,Pct VP Trips,TU Messages Per Minute,N TU Trips,Daily TU Trips,Pct TU Trips
60,2026-11-01,Western Contra Costa Transit Authority,04 - Bay Area / Oakland,Bay Area 511 WestCAT VehiclePositions,Bay Area 511 WestCAT TripUpdates,4568,Weekday,326.30,2520.50,11,14,40,218,5429.10,2.90,2766,197.60,60.20,2.70,2802,200.10,60.50
27,2026-11-01,Sonoma-Marin Area Rail Transit District,04 - Bay Area / Oakland,Bay Area 511 Sonoma-Marin Area Rail Transit VehiclePositions,Bay Area 511 Sonoma-Marin Area Rail Transit TripUpdates,624,Weekday,44.60,858.00,1,14,2,28,624.00,3.00,629,44.90,100.00,3.00,637,45.50,100.00
42,2026-11-01,Sonoma County Transit Schedule,04 - Bay Area / Oakland,Bay Area 511 Sonoma County Transit Vehicle Positions,Bay Area 511 Sonoma County Transit TripUpdates,2277,Weekday,227.70,2208.60,26,10,63,1008,10910.70,2.70,2243,224.30,98.20,2.00,2374,237.40,98.80
48,2026-11-01,Solano Transportation Authority,04 - Bay Area / Oakland,Bay Area 511 SolTrans VehiclePositions,Bay Area 511 SolTrans TripUpdates,4360,Weekday,311.40,3332.20,15,14,45,307,6065.90,3.00,4147,296.20,94.80,3.00,4371,312.20,98.70
33,2026-11-01,Santa Clara Valley Transportation Authority,04 - Bay Area / Oakland,Bay Area 511 Santa Clara Transit VehiclePositions,Bay Area 511 Santa Clara Transit TripUpdates,51355,Weekday,3668.20,52707.30,67,14,243,3228,158738.80,3.00,45163,3225.90,87.60,2.90,51203,3657.40,97.60
39,2026-11-01,San Mateo County Transit District,04 - Bay Area / Oakland,Bay Area 511 SamTrans VehiclePositions,Bay Area 511 SamTrans Trip Updates,23153,Weekday,1653.80,21902.60,75,14,336,1853,58139.10,3.00,22588,1613.40,97.20,3.00,22660,1618.60,97.50
30,2026-11-01,San Francisco Bay Ferry and Oakland Alameda Water Shuttle Schedule,04 - Bay Area / Oakland,Bay Area 511 San Francisco Bay Ferry VehiclePositions,Bay Area 511 San Francisco Bay Ferry TripUpdates,2518,Weekday,179.90,1130.20,6,14,26,11,368.10,3.00,2445,174.60,96.90,3.00,2532,180.90,99.40
9,2026-11-01,Peninsula Corridor Joint Powers Board,04 - Bay Area / Oakland,Bay Area 511 Caltrain VehiclePositions,Bay Area 511 Caltrain TripUpdates,1456,Weekday,104.00,1809.20,4,14,13,58,1989.00,2.50,1351,96.50,92.20,2.60,1361,97.20,92.70
57,2026-11-01,Napa Valley Transportation Authority,04 - Bay Area / Oakland,Bay Area 511 Vine Transit VehiclePositions,Bay Area 511 Vine Transit Trip Updates,3689,Weekday,263.50,3181.80,14,14,29,292,5889.50,2.90,2866,204.70,77.50,3.00,3597,256.90,96.80
21,2026-11-01,Marin County Transit District,04 - Bay Area / Oakland,Bay Area 511 Marin VehiclePositions,Bay Area 511 Marin TripUpdates,8115,Weekday,579.60,5880.70,20,14,54,543,14876.90,2.60,6246,446.10,76.70,2.60,8195,585.40,99.70


In [41]:
gtfs_table = utils.create_operator_table(
        operator_df, district_col = "Caltrans District"
    ).drop(columns = "Caltrans District")

In [42]:
gtfs_table

,Operator,Daily Trips,# Routes,# Shapes,# Stops,Daily Arrivals,Arrivals per Stop
0,Alameda-Contra Costa Transit District,4766,123,314,4679,217310,46.40
1,Central Contra Costa Transit Authority,866,42,89,1212,22538,18.60
2,City and County of San Francisco,8803,68,279,3239,325631,100.50
3,City of Fairfield,159,5,11,124,2484,20.00
4,City of Petaluma,221,11,20,152,3001,19.70
5,City of Santa Rosa,420,15,33,400,10406,26.00
6,City of Union City,251,6,16,167,6503,38.90
7,City of Vacaville,158,5,12,190,3544,18.70
8,Eastern Contra Costa Transit Authority,322,11,22,408,10775,26.40
9,"Golden Gate Bridge, Highway and Transportation District",256,10,23,246,6700,27.20


## `Legislative District Prep`

### sjoin_shapes_legislative_districts

In [50]:
from update_vars import DIGEST_DICT, PROCESSED_GCS, RAW_GCS, SHARED_GCS, abbrev_month

In [54]:
monthly_routes = gpd.read_parquet(
        f"{RAW_GCS}{DIGEST_DICT.route_map}_{abbrev_month}.parquet",
        storage_options={"token": credentials.token},
    columns=["name", "geometry"],
    )

In [48]:
monthly_routes.head(1).drop(columns = ["geometry"])

,name,year,month,month_first_day,route_name,direction_id,route_type,shape_id,shape_array_key,n_trips
0,Santa Maria Schedule,2026,10,2026-10-01,8__Rt 8. Tanglewood to Crossroads Shopping Center via McCoy Ln.,NaN,3,91b9279f-8025-48cc-9ce1-a3a907c02903,17c6f6ff7ef3837cdebe31e68db90e95,24


In [51]:
legislative_districts = gpd.read_parquet(
        f"{SHARED_GCS}legislative_districts.parquet",
        storage_options={"token": credentials.token},
    )

In [53]:
legislative_districts.sample().drop(columns = ["geometry"])

,legislative_district
8,AD 09


In [55]:
crosswalk = (
        gpd.sjoin(
            monthly_routes, legislative_districts, how="inner", predicate="intersects"
        )[["name", "legislative_district"]]
        .drop_duplicates()
        .sort_values(["name", "legislative_district"])
        .reset_index(drop=True)
    )

In [56]:
crosswalk.head(2)

,name,legislative_district
0,AC Transit Schedule,AD 14
1,AC Transit Schedule,AD 15


In [58]:
(
        f"{PROCESSED_GCS}{DIGEST_DICT.crosswalk_legislative}.parquet"
    )

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/crosswalk_transit_operators_legislative_districts.parquet'

In [60]:
crosswalk_url = (
        f"{PROCESSED_GCS}{DIGEST_DICT.crosswalk_legislative}.parquet"
    )

legislative_districts_list = (
        pd.read_parquet(
            crosswalk_url,
            columns=["legislative_district"],
            filesystem=gcsfs.GCSFileSystem(),
        )
        .legislative_district.unique()
        .tolist()
    )
